# 05 – Pipeline & Train/Test Split (BRFSS Diabetes)

Notebook này thực hiện chia tập phân tầng 80/20 (seed 42) và xây dựng pipeline scikit-learn. Các bước học tham số (imputation, winsorization, one-hot và scaling) nằm trong pipeline để được fit chỉ trên tập train; test được giữ riêng.

**Schema:** nhãn `Diabetes_binary` (0/1), 21 biến đầu vào. Dữ liệu trùng được giữ như trong notebook nguồn. Không đánh giá holdout test trong notebook này.

Chạy từ thư mục gốc dự án hoặc mở notebook trong `notebooks/05_Pipeline_Split`. Dependencies: pandas, numpy, scikit-learn.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

TARGET = "Diabetes_binary"
RANDOM_STATE = 42
TEST_SIZE = 0.20
EXPECTED_COLUMNS = [
    "HighBP", "HighChol", "CholCheck", "BMI", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "GenHlth",
    "MentHlth", "PhysHlth", "DiffWalk", "Sex", "Age", "Education",
    "Income", TARGET,
]

def find_project_root():
    # Notebook output directories are expected under <project>/notebooks/<stage>/.
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in candidates:
        if (base / "DATA").is_dir() or (base / "sources").is_dir():
            return base
    raise FileNotFoundError(
        "Không tìm thấy thư mục DATA/ hoặc sources/. Hãy mở notebook trong dự án "
        "hoặc chạy kernel từ một thư mục con của dự án."
    )

ROOT = find_project_root()
data_candidates = [
    ROOT / "DATA" / "raw" / "diabetes.csv",
    ROOT / "DATA" / "processed" / "diabetes_dedup.csv",
    ROOT / "sources" / "diabetes.csv",
]
DATA_PATH = next((p for p in data_candidates if p.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Không tìm thấy diabetes.csv. Đặt dữ liệu tại DATA/raw/diabetes.csv, "
        "DATA/processed/diabetes_dedup.csv hoặc sources/diabetes.csv."
    )
df = pd.read_csv(DATA_PATH)
missing = sorted(set(EXPECTED_COLUMNS) - set(df.columns))
if missing:
    raise ValueError(f"Thiếu cột theo schema BRFSS hiện tại: {missing}")
if not set(df[TARGET].dropna().unique()).issubset({0, 1}):
    raise ValueError(f"{TARGET} cần chứa nhãn nhị phân 0/1.")
print(f"Project: {ROOT}")
print(f"Dữ liệu: {DATA_PATH.relative_to(ROOT)}")
print(f"Kích thước: {df.shape[0]:,} dòng × {df.shape[1]} cột")
print(f"Số ô thiếu: {df.isna().sum().sum():,}; dòng trùng: {df.duplicated().sum():,}")
print("Phân bố nhãn: ")
print(df[TARGET].value_counts(dropna=False).sort_index())
df.head()


Project: c:\Users\Welcome\Desktop\ML cuối kỳ\MAY_HOC_NHOM\Do_An_Cuoi_Ky
Dữ liệu: DATA\raw\diabetes.csv
Kích thước: 253,680 dòng × 22 cột
Số ô thiếu: 0; dòng trùng: 24,206
Phân bố nhãn: 
Diabetes_binary
0    218334
1     35346
Name: count, dtype: int64


,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_binary
0,1,1,1,40,1,0,0,0,0,1,...,0,5,18,15,1,0,9,4,3,0
1,0,0,0,25,1,0,0,1,0,0,...,1,3,0,0,0,0,7,6,1,0
2,1,1,1,28,0,0,0,0,1,0,...,1,5,30,30,1,0,9,4,8,0
3,1,0,1,27,0,0,0,1,1,1,...,0,2,0,0,0,0,11,3,6,0
4,1,1,1,24,0,0,0,1,1,1,...,0,2,3,0,0,0,11,5,4,0


## 1. Chia train/test

`stratify=y` giữ tỷ lệ lớp gần như tương tự ở hai tập. Không xóa trùng hay tạo đặc trưng trước chia; split này là bước đầu tiên sau khi nạp và kiểm tra schema.

In [2]:
from sklearn.model_selection import train_test_split, StratifiedKFold

X = df.drop(columns=TARGET).copy()
y = df[TARGET].astype(int).copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {X_train.shape}; test: {X_test.shape}")
print("Tỷ lệ lớp train:", y_train.value_counts(normalize=True).sort_index().round(4).to_dict())
print("Tỷ lệ lớp test :", y_test.value_counts(normalize=True).sort_index().round(4).to_dict())


Train: (202944, 21); test: (50736, 21)
Tỷ lệ lớp train: {0: 0.8607, 1: 0.1393}
Tỷ lệ lớp test : {0: 0.8607, 1: 0.1393}


## 2. Các transformer trong pipeline

BMI được winsorize theo phân vị 1%–99%, với cận chỉ được học khi pipeline fit trên train. Ordinal fields có thể giữ số hoặc one-hot; imputation median và scaling cũng nằm bên trong pipeline. Pipeline dưới đây kết thúc ở bước tiền xử lý để có thể tái sử dụng với estimator khác.

In [3]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ORDINAL = ["Age", "Education", "Income", "GenHlth"]

class Winsorizer(BaseEstimator, TransformerMixin):
    """Cắt BMI về các phân vị lower/upper đã ước lượng trong fit."""
    def __init__(self, columns=("BMI",), lower=0.01, upper=0.99):
        self.columns = columns
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        self.bounds_ = {
            col: (X[col].quantile(self.lower), X[col].quantile(self.upper))
            for col in self.columns
        }
        return self

    def transform(self, X):
        out = X.copy()
        for col, (low, high) in self.bounds_.items():
            out[col] = out[col].clip(low, high)
        return out

class HealthFeatureEngineer(BaseEstimator, TransformerMixin):
    """Tạo các đặc trưng bổ sung, giữ các cột gốc."""
    def __init__(self, interaction=True):
        self.interaction = interaction

    def fit(self, X, y=None):
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def transform(self, X):
        out = X.copy()
        out["BMI_WHO_Group"] = pd.cut(
            out["BMI"], [-np.inf, 18.5, 25, 30, np.inf], labels=False, right=False
        ).astype(float)
        out["Comorbidity_Count"] = out[["HighBP", "HighChol", "Stroke", "HeartDiseaseorAttack"]].sum(axis=1)
        out["Healthy_Lifestyle_Score"] = out[["PhysActivity", "Fruits", "Veggies"]].sum(axis=1)
        if self.interaction:
            out["Age_x_HighBP"] = out["Age"] * out["HighBP"]
        return out

    def get_feature_names_out(self, input_features=None):
        names = list(self.feature_names_in_) + ["BMI_WHO_Group", "Comorbidity_Count", "Healthy_Lifestyle_Score"]
        if self.interaction:
            names.append("Age_x_HighBP")
        return np.asarray(names, dtype=object)

def build_preprocessor(onehot_ordinal=False, add_engineered=True):
    """Trả về pipeline tiền xử lý; mọi fit chỉ dùng dữ liệu truyền vào fit()."""
    steps = [("imputer", SimpleImputer(strategy="median").set_output(transform="pandas")),
             ("winsor", Winsorizer(columns=("BMI",)))]
    steps.append(("features", HealthFeatureEngineer() if add_engineered else "passthrough"))
    if onehot_ordinal:
        encoder = OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False)
        columns = ColumnTransformer(
            [("ordinal", encoder, ORDINAL)], remainder="passthrough",
            verbose_feature_names_out=False
        ).set_output(transform="pandas")
        steps.append(("onehot", columns))
    steps.append(("scaler", StandardScaler()))
    return Pipeline(steps)


## 3. Fit trên train và chuyển đổi test

Mặc định giữ các biến thứ tự dạng số. Có thể bật `onehot_ordinal=True` cho phép so sánh biểu diễn one-hot. Việc gọi `transform` trên test không fit lại bất kỳ bước nào.

In [4]:
preprocessor = build_preprocessor(onehot_ordinal=False, add_engineered=True)
X_train_ready = preprocessor.fit_transform(X_train, y_train)
X_test_ready = preprocessor.transform(X_test)
print("Train sau biến đổi:", X_train_ready.shape)
print("Test sau biến đổi :", X_test_ready.shape)
print("Cận winsor BMI (học từ train):", preprocessor.named_steps["winsor"].bounds_)
print("Max |mean| train sau scaling:", float(np.abs(X_train_ready.mean(axis=0)).max()))
print("Max |mean| test sau scaling :", float(np.abs(X_test_ready.mean(axis=0)).max()))


Train sau biến đổi: (202944, 25)
Test sau biến đổi : (50736, 25)
Cận winsor BMI (học từ train): {'BMI': (np.float64(18.0), np.float64(50.0))}
Max |mean| train sau scaling: 2.415811690291258e-16
Max |mean| test sau scaling : 0.010181476350562945


## 4. Pipeline hoàn chỉnh có mô hình (tùy chọn)

Đoạn này tạo một pipeline huấn luyện được. Khi dùng cross-validation, hãy truyền pipeline nguyên vẹn vào `cross_validate` để mỗi fold tự fit imputer, winsorizer, encoder, scaler và mô hình. Tập test vẫn chưa được đánh giá.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer, f1_score

model_pipeline = Pipeline([
    ("preprocessing", build_preprocessor(onehot_ordinal=False, add_engineered=True)),
    ("model", LogisticRegression(
        class_weight="balanced", solver="liblinear", max_iter=500,
        random_state=RANDOM_STATE
    )),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print("Pipeline đã sẵn sàng:", model_pipeline)
# Để chạy đánh giá CV chỉ trên train, bỏ dấu # ở các dòng dưới:
# scores = cross_validate(
#     model_pipeline, X_train, y_train, cv=cv, n_jobs=1,
#     scoring={"macro_f1": make_scorer(f1_score, average="macro"), "accuracy": "accuracy"}
# )
# print({k: (float(v.mean()), float(v.std())) for k, v in scores.items() if k.startswith("test_")})


Pipeline đã sẵn sàng: Pipeline(steps=[('preprocessing',
                 Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                                 ('winsor', Winsorizer()),
                                 ('features', HealthFeatureEngineer()),
                                 ('scaler', StandardScaler())])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=500,
                                    random_state=42, solver='liblinear'))])


## Ghi chú

- Split: stratified 80/20, seed 42; holdout test không dùng trong chọn mô hình/CV.
- Pipeline per-fold fit ngăn rò rỉ thống kê của imputation, winsorization, encoding và scaling.
- Dùng đúng nhãn `Diabetes_binary` của CSV. Nếu nhóm chuyển sang `Diabetes_012`, cần đổi target và cập nhật báo cáo/schema thống nhất.